In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

parquet_path = "../data/processed/criteo-uplift-v2.1.parquet"

In [ ]:
# Load train and validation only. Test stays locked until the end.
df = pd.read_parquet(parquet_path, filters=[('split', '!=', 'test')])
features = [f'f{i}' for i in range(12)]

# Fit on the 1.5M dev rows, score on validation.
dev = df[df['dev']]
val = df[df['split']=='validation']
print(f'Dev rows: {len(dev)}, validation rows: {len(val)}')

In [ ]:
# Same LightGBM settings for every model, so the learners are compared fairly.
params = dict(n_estimators=200, learning_rate=0.05, num_leaves=31, min_child_samples=100, random_state=42, verbose=-1)

In [ ]:
# T-learner: one model for the ad group, one for the no-ad group.
# Uplift = predicted visit chance with the ad minus without it.
dev_ad = dev[dev['treatment']==1]
dev_no_ad = dev[dev['treatment']==0]
t_model_ad = lgb.LGBMClassifier(**params).fit(dev_ad[features], dev_ad['visit'])
t_model_no_ad = lgb.LGBMClassifier(**params).fit(dev_no_ad[features], dev_no_ad['visit'])

uplift_t = t_model_ad.predict_proba(val[features])[:, 1] - t_model_no_ad.predict_proba(val[features])[:, 1]

In [ ]:
# S-learner: one model with treatment as a feature.
# Uplift = its prediction with treatment set to 1 minus with treatment set to 0.
s_model = lgb.LGBMClassifier(**params).fit(dev[features + ['treatment']], dev['visit'])

uplift_s = (s_model.predict_proba(val[features].assign(treatment=1))[:, 1]
            - s_model.predict_proba(val[features].assign(treatment=0))[:, 1])

In [ ]:
# Random baseline: uplift scores with no information. Every model should beat this.
rng = np.random.default_rng(42)
uplift_random = rng.random(len(val))

In [ ]:
# Uplift scores on validation, one column per model. Step 3 evaluates these.
val_scores = val[['treatment', 'visit', 'conversion']].assign(
    uplift_t=uplift_t,
    uplift_s=uplift_s,
    uplift_random=uplift_random,
)

# Quick sanity check: the average predicted uplift should be near the validation ATE.
val_ate = val[val['treatment']==1]['visit'].mean() - val[val['treatment']==0]['visit'].mean()
print(f'Validation ATE for visit: {val_ate:.6f}')
print(f'Mean predicted uplift, T-learner: {uplift_t.mean():.6f}')
print(f'Mean predicted uplift, S-learner: {uplift_s.mean():.6f}')
print(f'S-learner rows with zero uplift: {(uplift_s == 0).mean():.1%}')